# 🚀 BoneRAG — FracAtlas Dataset BiomedCLIP GPU Indexing

Nối tiếp nghiên cứu **BoneRAG (Medical Visual Question Answering using Image RAG)**, notebook này tự động hóa toàn bộ quá trình mã hóa (encoding) tập dữ liệu **FracAtlas (4,082 ảnh X-quang xương)** bằng mô hình **Microsoft BiomedCLIP-PubMedBERT** trên Google Colab T4 GPU miễn phí.

### 📌 Các bước thực hiện:
1. **Bật GPU**: Menu `Runtime` -> `Change runtime type` -> Chọn `T4 GPU`.
2. **Chạy tất cả các ô (Ctrl + F9)**.
3. Sau khi chạy xong, tải 2 file kết quả `fracatlas_biomedclip.faiss` và `fracatlas_metadata.json` ở thanh thư mục bên trái về máy.

In [ ]:
# [Step 1] Cài đặt các thư viện cần thiết trên GPU
!pip install -q torch open_clip_torch faiss-cpu pillow qdrant-client tqdm huggingface_hub

In [ ]:
# [Step 2] Kiểm tra thiết bị GPU & Tải mô hình BiomedCLIP (Microsoft)
import torch
import open_clip
import faiss
import json
import os
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Thiết bị đang sử dụng: {device}")
if device == "cuda":
    print(f"🎮 Tên GPU: {torch.cuda.get_device_name(0)}")

print("\n📦 Đang tải BiomedCLIP-PubMedBERT (Microsoft)...")
model_name = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
model, _, preprocess = open_clip.create_model_and_transforms(model_name)
tokenizer = open_clip.get_tokenizer(model_name)
model.to(device).eval()
print("✅ Mô hình BiomedCLIP đã sẵn sàng!")

In [ ]:
# [Step 3] Clone Dataset FracAtlas về môi trường Colab
print("📥 Đang clone tập dữ liệu FracAtlas...")
!git clone --depth 1 https://github.com/huyen-nguyen/FracAtlas.git ./fracatlas_repo || true

image_files = list(Path("./fracatlas_repo").rglob("*.jpg")) + list(Path("./fracatlas_repo").rglob("*.png"))
print(f"📷 Đã tìm thấy {len(image_files)} tệp ảnh X-quang trong FracAtlas!")

In [ ]:
# [Step 4] Mã hóa 4,082 ảnh X-quang qua BiomedCLIP trên GPU
vectors = []
metadata = []

print(f"🚀 Bắt đầu trích xuất Vector Embedding 512-dim cho {len(image_files)} ảnh X-quang...")
for img_path in tqdm(image_files, desc="BiomedCLIP Encoding"):
    try:
        img = Image.open(img_path).convert("RGB")
        with torch.no_grad():
            tensor = preprocess(img).unsqueeze(0).to(device)
            feat = model.encode_image(tensor)
            feat /= feat.norm(dim=-1, keepdim=True)
            vectors.append(feat[0].cpu().numpy().astype(np.float32))

        is_frac = "fractured" in img_path.name.lower() or "fracture" in str(img_path.parent).lower()
        metadata.append({
            "image_id": f"fracatlas-{"fractured" if is_frac else "normal"}-{img_path.stem.lower()}",
            "title": f"FracAtlas X-ray {img_path.name}",
            "body_part": "forearm/wrist",
            "diagnosis": "fracture" if is_frac else "normal",
            "fracture_type": "fractured" if is_frac else "none",
            "region": "forearm and wrist",
            "evidence_note": f"FracAtlas real X-ray dataset case {img_path.name}.",
            "text": f"fracatlas {"fractured" if is_frac else "normal"} xray wrist forearm bone case {img_path.stem.lower()}",
            "image_path": str(img_path)
        })
    except Exception as exc:
        continue

print(f"\n✅ Đã mã hóa thành công {len(vectors)} vector!")

In [ ]:
# [Step 5] Tạo FAISS Index 512-dim & Xuất file lưu trữ
vec_matrix = np.array(vectors, dtype=np.float32)
dim = vec_matrix.shape[1]

# Khởi tạo FAISS Inner Product Index (Cosine similarity cho normalized vectors)
index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(vec_matrix)
index.add(vec_matrix)

# Ghi ra file chỉ số FAISS và file Metadata JSON
faiss.write_index(index, "fracatlas_biomedclip.faiss")
with open("fracatlas_metadata.json", "w", encoding="utf-8") as fh:
    json.dump(metadata, fh, ensure_ascii=False, indent=2)

print("\n=======================================================")
print("🎉 HOÀN THÀNH XÂY CHỈ SỐ FAISS CỦA FRACATLAS!")
print(f"  • Kích thước Vector: {dim} dimensions")
print(f"  • Tổng số bản ghi Indexed: {index.ntotal}")
print("\n📁 Đã xuất 2 file tại thư mục Colab:")
print("  1. fracatlas_biomedclip.faiss  (File ma trận FAISS Index)")
print("  2. fracatlas_metadata.json     (File Metadata chi tiết)")
print("=======================================================")